In [ ]:
# ============================================
# CELL 1: Check GPU and Install Dependencies
# ============================================
import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

# Install required packages
!pip install -q diffusers transformers accelerate
!pip install -q fastapi uvicorn pyngrok python-multipart
!pip install -q opencv-python-headless
!pip install -q git+https://github.com/facebookresearch/detectron2.git

print("✓ Dependencies installed!")

In [ ]:
# ============================================
# CELL 2: Download CatVTON Model
# ============================================
from huggingface_hub import snapshot_download

# Create directories
!mkdir -p models/catvton

# Download CatVTON model (this takes 5-10 minutes first time)
print("Downloading CatVTON model...")
snapshot_download(
    repo_id="zhengchong/CatVTON",
    local_dir="./models/catvton",
    local_dir_use_symlinks=False
)

print("✓ Model downloaded!")

In [ ]:
# ============================================
# CELL 3: Clone CatVTON Repository
# ============================================
!git clone https://github.com/Zheng-Chong/CatVTON.git
%cd CatVTON

# Download preprocessor models
print("Downloading preprocessing models...")
!python scripts/download_models.py

%cd ..
print("✓ Setup complete!")

In [ ]:
# ============================================
# CELL 4: Create Simplified VTON Pipeline
# ============================================
%%writefile vton_simple.py

import torch
from PIL import Image
from diffusers import AutoPipelineForImage2Image
from transformers import CLIPVisionModelWithProjection
import numpy as np


class SimplifiedVTON:
    def __init__(self, model_path="./models/catvton", device="cuda"):
        self.device = device
        self.dtype = torch.float16 if device == "cuda" else torch.float32

        print("Loading CatVTON pipeline...")
        # Load the pipeline components
        from diffusers import StableDiffusionPipeline
        self.pipe = StableDiffusionPipeline.from_pretrained(
            model_path,
            torch_dtype=self.dtype,
            safety_checker=None
        ).to(device)

        self.pipe.enable_attention_slicing()
        if device == "cuda":
            self.pipe.enable_xformers_memory_efficient_attention()

        print("✓ Pipeline loaded!")

    def process(self, person_image_path, garment_image_path, output_path):
        """Simple inference wrapper"""
        # Load images
        person_img = Image.open(person_image_path).convert("RGB")
        garment_img = Image.open(garment_image_path).convert("RGB")

        # Resize to model input size
        person_img = person_img.resize((512, 768))
        garment_img = garment_img.resize((512, 768))

        # Create prompt
        prompt = "high quality, detailed, photorealistic"

        # Run inference
        result = self.pipe(
            prompt=prompt,
            image=person_img,
            num_inference_steps=50,
            guidance_scale=7.5
        ).images[0]

        # Save result
        result.save(output_path)
        return result


print("✓ SimplifiedVTON class created!")

In [ ]:
# ============================================
# CELL 5: Create FastAPI Server
# ============================================
%%writefile api_server.py

from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
from PIL import Image
import uuid
from pathlib import Path

# Import our VTON class
from vton_simple import SimplifiedVTON

# Initialize FastAPI
app = FastAPI(title="VTON API")

# Enable CORS
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Global model instance
vton_model = None


@app.on_event("startup")
async def startup():
    global vton_model
    print("Loading VTON model...")
    vton_model = SimplifiedVTON()
    print("✓ Model ready!")


@app.get("/")
async def root():
    return {"status": "online", "service": "VTON API"}


@app.get("/health")
async def health():
    return {
        "status": "healthy",
        "model_loaded": vton_model is not None
    }


@app.post("/try-on")
async def try_on(
        person_image: UploadFile = File(...),
        garment_image: UploadFile = File(...)
):
    """Virtual try-on endpoint"""

    try:
        # Generate unique ID
        request_id = str(uuid.uuid4())

        # Create temp directory
        temp_dir = Path("temp")
        temp_dir.mkdir(exist_ok=True)

        # Save uploaded images
        person_path = temp_dir / f"{request_id}_person.jpg"
        garment_path = temp_dir / f"{request_id}_garment.jpg"
        result_path = temp_dir / f"{request_id}_result.png"

        # Write files
        person_path.write_bytes(await person_image.read())
        garment_path.write_bytes(await garment_image.read())

        # Run inference
        print(f"Processing {request_id}...")
        vton_model.process(
            str(person_path),
            str(garment_path),
            str(result_path)
        )

        # Return result
        return FileResponse(
            result_path,
            media_type="image/png",
            headers={"X-Request-ID": request_id}
        )

    except Exception as e:
        return {"error": str(e)}, 500


# Run server
if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

In [ ]:
# ============================================
# CELL 6: Setup Ngrok Tunnel
# ============================================
!pip install -q pyngrok

from pyngrok import ngrok
import getpass

# Get your ngrok authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
print("Get your authtoken from: https://dashboard.ngrok.com/get-started/your-authtoken")
ngrok_token = getpass.getpass("Enter your ngrok authtoken: ")

# Set authtoken
ngrok.set_auth_token(ngrok_token)

print("✓ Ngrok configured!")

In [ ]:
# ============================================
# CELL 7: Start API Server with Tunnel
# ============================================
import threading
import time
import uvicorn
from pyngrok import ngrok

# Start ngrok tunnel
public_url = ngrok.connect(8000)
print("\n" + "=" * 60)
print(f"🌐 PUBLIC API URL: {public_url}")
print("=" * 60)
print("\nCopy this URL to use in your frontend!")
print(f"Example: {public_url}/try-on")
print("\n")


# Start FastAPI server in background thread
def run_server():
    import api_server
    uvicorn.run(api_server.app, host="0.0.0.0", port=8000)


server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to start
time.sleep(10)

print("✓ Server is running!")
print(f"Test it: curl {public_url}/health")

In [ ]:
# ============================================
# CELL 8: Test the API
# ============================================
import requests
from IPython.display import Image as IPImage, display

# Your public URL from above
API_URL = "YOUR_NGROK_URL_HERE"  # Replace with actual URL from Cell 7

# Test health endpoint
response = requests.get(f"{API_URL}/health")
print("Health check:", response.json())

# Upload test images and try-on
# (You can upload images to Colab using the file browser on the left)

In [ ]:
# ============================================
# CELL 9: Upload Test Images (Optional)
# ============================================
from google.colab import files
import shutil

print("Upload person image:")
person_upload = files.upload()
person_filename = list(person_upload.keys())[0]
shutil.move(person_filename, "test_person.jpg")

print("\nUpload garment image:")
garment_upload = files.upload()
garment_filename = list(garment_upload.keys())[0]
shutil.move(garment_filename, "test_garment.jpg")

print("✓ Test images uploaded!")

In [ ]:
# ============================================
# CELL 10: Run Test Try-On
# ============================================
import requests
from IPython.display import Image as IPImage, display

API_URL = "YOUR_NGROK_URL"  # From Cell 7

# Test the try-on
files = {
    'person_image': open('test_person.jpg', 'rb'),
    'garment_image': open('test_garment.jpg', 'rb')
}

print("Sending try-on request...")
response = requests.post(f"{API_URL}/try-on", files=files)

if response.status_code == 200:
    # Save and display result
    with open('result.png', 'wb') as f:
        f.write(response.content)

    print("✓ Try-on complete!")
    display(IPImage('result.png'))
else:
    print(f"Error: {response.status_code}")
    print(response.text)